# Extrapolate bias-corrected salinity from EN4 grid to ISMIP
Find salinity (bias-corrected) in fjords according to lookup table of effective depth.  Write to 1km x 1km ISMIP grid.

19 Mar 2026 | EHU

### Setup

In [ ]:
## files
SelModel = 'CESM2-WACCM' ## model label for input and output
PressureAlreadyIncluded = True ## was the pressure effect accounted for in Step 1?
#YearTags = ['1850', '1860']

DirSave = f'/Users/eultee/Desktop/'
model_file = f'{DirSave}/soQDM_additive-AllLevels-CommonGrid-CESM2-WACCM-1850_2014-20260320.nc'

# Output options
output_option = 'single' # choose 'single' or 'batch'
# If batch is chosen, the code below will automatically split up the fill model dataset into
# time spans, as defined by the two variables below, and will write separate netcdfs for each.
time_slice_hist = slice('1850', '2014')
time_slice_proj = slice('2015', '2100')


### Imports

In [ ]:
# libraries
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import netCDF4 as nc
from netCDF4 import Dataset
import os
from pyproj import Transformer
from scipy.interpolate import interp1d
from datetime import datetime


In [ ]:
## load effective geometry (spatial look-up table)

# files
xy_eff_file = '../oceanTF/XY_eff.nc'
z_eff_file = '../oceanTF/z_eff.nc'

# effective geometry
# NB replace masked values with NaNs
X_eff = nc.Dataset(xy_eff_file).variables['X_eff'][:].filled(np.nan)
Y_eff = nc.Dataset(xy_eff_file).variables['Y_eff'][:].filled(np.nan)
z_eff = nc.Dataset(z_eff_file).variables['z_eff'][:].filled(np.nan)

# ismip coordinates at which effective geometry applies
x = nc.Dataset(xy_eff_file).variables['x'][:].filled(np.nan)
y = nc.Dataset(xy_eff_file).variables['y'][:].filled(np.nan)
X, Y = np.meshgrid(x, y)

# vertical grid of effective depths
z = np.flipud(np.unique(z_eff[z_eff<=0]))


In [ ]:
fig, ax= plt.subplots()
sc = ax.scatter(X_eff[::4, ::4], Y_eff[::4, ::4], c=z_eff[::4,::4])
plt.colorbar(sc, label='effective depth read in (m)')

Use xarray to load in a view of the dataset for future reference.

In [ ]:
## load and process data
ds_model = xr.open_dataset(model_file, decode_times='timeDim')

# cmip model coordinates
lat_model = nc.Dataset(model_file).variables['lat'][:].filled(np.nan)
lon_model_raw = nc.Dataset(model_file).variables['lon'][:].filled(np.nan)
z_model = (nc.Dataset(model_file).variables['depth'][:].filled(np.nan)) ## QDM processed file should have depth in m
t_model = nc.Dataset(model_file).variables['time'][:].filled(np.nan)

## Common grid has y, x in EPSG 3413 already
x_model = nc.Dataset(model_file).variables['x'][:].filled(np.nan)
y_model = nc.Dataset(model_file).variables['y'][:].filled(np.nan)

# lon_model = np.mod((lon_model_raw+180), 360) -180 ## put back on -180 to 180 axis
lon_model = lon_model_raw
# lon_grid, lat_grid = np.meshgrid(lon_model, lat_model) ## necessary if 1D arrays


# # get coordinates in EPSG:3413 for consistency with ismip grid
# mapping = Transformer.from_crs("epsg:4326", "epsg:3413", always_xy=True)
# x_model, y_model = mapping.transform(lon_model, lat_model)

# ## load so
## No masking, just set <0 to 0 and allow all other missing data -- should be missing where there is no ocean data in domain
#so_model_raw = nc.Dataset(model_file).variables['so'][:]
so_model_raw_ds = xr.open_dataset(model_file)
so_model_raw_ds = so_model_raw_ds.transpose('depth','time','y','x')
so_model_raw = so_model_raw_ds['so'].to_masked_array()
so_model_nonneg = np.ma.masked_less(so_model_raw, 0) ## filter out erroneous negative values

so_model = so_model_raw
so_model[so_model<0] = 0

In [ ]:
## we can tell from the read-in above that so_model from EN4 has dimensions (depth, lat, lon, time)
## change to ## depth, time, lat, lon as the code is expecting

## for CESM2-WACCM raw, it is (time, depth, lat, lon)
## for CESM2-WACCM QDM-corrected, it is (depth, nlat, nlon, time)
## EN4 is (time, depth, y, x)

## make a filled version of the masked array for interpolation and shuffling
so_mod_filled = np.ma.filled(so_model, fill_value=np.nan) ## filter out fill value
so_nop_model_orig_crop = so_mod_filled.reshape(so_mod_filled.shape[0], ## depth
                                          so_mod_filled.shape[1], ## time?
                                          so_mod_filled.shape[2]*so_mod_filled.shape[3]) ## should be x*y
so_nop_model_orig_crop = np.ma.masked_greater(so_nop_model_orig_crop, 1.1e10)

# # interpolate the salinity from cmip vertical grid onto the effective depth grid
# interp_so = interp1d(z_model, so_nop_model_orig_crop, axis=0, bounds_error=False, fill_value='extrapolate')
# ## note which axis is depth/z. Donald's default had axis=1, but checking the shape of so_model 
# ## showed that it had depth as its zeroth axis. Consistent with so_shuffled order above

# ## make z increase downwards -- consistent with CESM2-WACCM convention
# z_inc_down = -1*z
# so_nop_model = interp_so(z_inc_down)
## ---

xm, ym = np.meshgrid(x_model, y_model)
x_mod = xm.flatten()
y_mod = ym.flatten()

# get max depth at which each cmip point has a thermal forcing value
depth_model = np.full(x_mod.shape, np.nan)
for i in range(len(x_mod)):
    so = so_nop_model_orig_crop[:,0,i] ## careful with order of axes here
    if np.all(np.isnan(so)):
        continue
    elif np.all(~np.isnan(so)):
        depth_model[i] = z[-1]
    else:
        first_nan = np.argmax(np.isnan(so))
        depth_model[i] = z[first_nan-1]
        

In [ ]:
## plots to check things make sense

fig, ax = plt.subplots()
# ax.contourf(lon_model, lat_model, so_model[0,0, :,:]) ## for raw
ax.contourf(lon_model, lat_model, so_model[0,0,:,:]) ## for QDM-corrected
plt.show()

# plt.scatter(xm, ym, c=so_nop_model[0,0,:])
# plt.colorbar(label='thermal forcing at first time and depth slice (°C)')
# plt.axis('equal')
# plt.show()

plt.scatter(x_mod, y_mod, c=depth_model)
plt.colorbar(label='max depth of non-NaN data in CMIP model (m)')
plt.axis('equal')
plt.show()


In [ ]:
## match ismip points to cmip points

# linear versions of effective position arrays
X_eff_flat = X_eff.flatten()
Y_eff_flat = Y_eff.flatten()

# initialise arrays that will store the linear index of the required cmip point
x_ind = np.full_like(X_eff_flat, np.nan, dtype=float)
z_ind = np.full_like(X_eff_flat, np.nan, dtype=float)

# loop over effective depth grid
for k, z_val in enumerate(z):
    
    # get all ismip points with this effective depth
    i_inds = np.where(z_eff.flatten() == z_val)[0]

    # save the effective depth index
    z_ind[i_inds] = k

    # get all cmip points that have data at this depth
    c_inds = np.where(depth_model.flatten() <= z_val)[0]

    # loop over ismip points with this effective depth and find closest cmip point to effective position
    for i in i_inds:
        dsq = (x_mod[c_inds] - X_eff_flat[i])**2 + (y_mod[c_inds] - Y_eff_flat[i])**2
        id_min = np.argmin(dsq)
        x_ind[i] = c_inds[id_min]
    

In [ ]:
## now we fill the ismip so using the matched cmip positions and depths

# initialise array
so_nop = np.full((len(t_model), len(x_ind)), np.nan)

# fill array knowing that the ismip so(t,i) equals the cmip so(z_ind(i),t,x_ind(i))
for i in range(len(x_ind)):
    if not np.isnan(x_ind[i]):
        ## depth first
        so_nop[:,i] = so_nop_model_orig_crop[int(z_ind[i]), :, int(x_ind[i])]

# reshape array for consistency with ismip coordinates
so_nop = so_nop.reshape(len(t_model), X.shape[0], X.shape[1])


In [ ]:
## plot an example of the result

plt.pcolormesh(X, Y, so_nop[0, :, :])
cbar = plt.colorbar()
cbar.set_label('extrapolated salinity at first time slice')
plt.axis('equal')
plt.show()

## Write out
Use xarray to write nice metadata and keep date format intact.  These cells prep the data to write, then there are two separate options to write a single or batched file.  The choice of `output_option` is set in the header of this notebook.

In [ ]:
so_out_ds = xr.Dataset(
    data_vars = dict(so=(['time', 'y', 'x'], so)), 
    coords = dict(
        time = ds_model.time, ## cheat a bit and take the datetime64 index from ds_model, read in above
        y = y,
        x = x)
)

so_out_ds

In [ ]:
so_out_ds = so_out_ds.convert_calendar('noleap')
so_out_ds

Explicitly set some metadata, and specify that so should be written as float32 (like Donald did in his approach).

In [ ]:
so_out_ds.coords['y'].attrs['units'] = 'meters'
so_out_ds.coords['y'].attrs['projection'] = 'EPSG:3413'
so_out_ds.coords['x'].attrs['units'] = 'meters'
so_out_ds.coords['x'].attrs['projection'] = 'EPSG:3413'

so_out_ds.so.attrs['long_name'] = 'Ocean salinity'
# so_out_ds.so.attrs['fill_value'] = 1.1e20 ## list fillvalue even though we have not done explicit filling in this step
so_out_ds

In [ ]:
so_reduced = so_out_ds.astype('float32')
so_reduced

---
### Single output file option
Use this if your input is already split by period, or you have a small output dataset.

In [ ]:
if output_option == 'single':
    ## SINGLE OUTPUT FILE
    ## write to output file
    from datetime import datetime
    
    now = datetime.now()
    
    ## Make filename tags showing time for the output 
    # ## this is very janky, but datetime64 objects are stubborn
    # FirstYear = so_reduced.time[0].values.astype(str).split('-')[0] 
    # LastYear = so_reduced.time[-1].values.astype(str).split('-')[0]
    ## with cftime axes:
    FirstYear = so_reduced.time[0].values.item().year
    LastYear = so_reduced.time[-1].values.item().year
    
    ## file name
    out_fn = DirSave + '/so_QDMCorrected-ISMIP_Grid-{}-{}_{}-{}.nc'.format(SelModel, 
                                                        FirstYear, LastYear,  
                                                        now.strftime('%Y%m%d'))
    
    # ds_temp = so_out.to_dataset(name='so')
    ds_out = so_reduced.assign_attrs(title='Ocean salinity for {}'.format(SelModel),
                                 summary='Salinity extracted from CMIP model, following Slater inland mapping,' + 
                                 'in a bounding box around Greenland, for ISMIP7 Greenland forcing.' +
                                    # ' Mean correction based on EN4, 1985-2014.' +
                                     'Additive QDM correction based on EN4, 1985-2014.' +
                                    ' Process code: github.com/ehultee/gris-iceocean-process',
                                 institution='NASA Goddard Space Flight Center',
                                 creation_date=now.strftime('%Y-%m-%d %H:%M:%S'))
    
    ## write it!
    ds_out.to_netcdf(path=out_fn,
                    encoding={'so': {'zlib': True, 'complevel':9}}) ## set compression level

---
### Batch writing option
Run these if you want to split by years.  Necessary if your input was several time periods grouped together, or you have a very large dataset.

In [ ]:
if output_option == 'batch':
    ## MULTI OUTPUT FILE
    # from dask.diagnostics import ProgressBar

    so_reduced_hist = so_reduced.sel(time=time_slice_hist)
    so_reduced_proj = so_reduced.sel(time=time_slice_proj)

    now = datetime.now()
    
    to_write = [so_reduced_hist,so_reduced_proj]
    
    for d in to_write:
        print('Processing...')
        ## Make filename tags showing time for the output 
        # ## this is very janky, but datetime64 objects are stubborn
        # FirstYear = d.time[0].values.astype(str).split('-')[0] 
        # LastYear = d.time[-1].values.astype(str).split('-')[0]
        ## with cftime axes:
        FirstYear = d.time[0].values.item().year
        LastYear = d.time[-1].values.item().year
        
        ## file name
        out_fn = DirSave + '/' + 'so_QDMCorrected-ISMIP_Grid-{}-{}_{}-{}.nc'.format(SelModel, 
                                                            FirstYear, LastYear, 
                                                            now.strftime('%Y%m%d'))
        print('Assigning attrs')
        # ds_temp = so_out.to_dataset(name='so')
        ds_out = d.assign_attrs(title='Ocean salinity {}'.format(SelModel),
                                     summary='Salinity extracted from CMIP model, following Slater inland mapping,' + 
                                 'in a bounding box around Greenland, for ISMIP7 Greenland forcing.' +
                                        'Additive QDM correction based on EN4, 1985-2014.' +
                                        ' Process code: github.com/ehultee/gris-iceocean-process',
                                     institution='NASA Goddard Space Flight Center',
                                     creation_date=now.strftime('%Y-%m-%d %H:%M:%S'))
    
        print('Writing: ' + out_fn)
        ## write it!
        # with ProgressBar():
        ds_out.to_netcdf(path=out_fn,
                        encoding={'so': {'zlib': True, 'complevel':9}}) ## set compression level
    
    print('Done!')